# Text Splitters for paragraph segmentation and preprocessing

In [19]:
import pandas as pd
import os

from pathlib import Path
import random

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [6]:
df = pd.read_csv("../data/snomed-ct-entity-linking-challenge-1.0.0/mimic-iv_notes_training_set.csv")

In [9]:
print(df.text.values[0])

 
Name:  ___                  Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: SURGERY
 
Allergies: 
Penicillins
 
Attending: ___.
 
Chief Complaint:
Biliary pancreatitis
 
Major Surgical or Invasive Procedure:
___: Laparoscopic cholecystectomy

 
History of Present Illness:
Mr. ___ is a ___ man who had severe biliary 
pancreatitis resulting in pancreatic necrosis for which he was 
treated with nasojejunal feedings and pancreatic rest.  He had 
initially had multisystem organ failure, which improved. Mr. 
___ has a large postnecrotic pseudocyst, which has been 
drained through a minimally invasive approach into his GI tract. 
 He has some debris, but this is not currently infected. The 
patient was followed by Dr. ___ in his ___ 
clinic to discuss cholecystectomy. After discussion of all 
risks, benefits and possible outcomes, patient was scheduled for 
elective cholecystectomy on ___.
 
Past Medical History:

In [11]:
import re

In [24]:
def format_headings_as_markdown(text, section_names):
    # Create a regex pattern to match any of the section names at the beginning of a line, case insensitive
    pattern = r'^(?:' + '|'.join(re.escape(name) for name in section_names) + r'):$'
    
    # Define a replacement function that adds "## " before the matched section name
    def replace_with_markdown_heading(match):
        return "## " + match.group(0)
    
    # Use re.sub to replace each matched section name with the markdown heading, case insensitive
    formatted_text = re.sub(pattern, replace_with_markdown_heading, text, flags=re.MULTILINE | re.IGNORECASE)
    
    # Insert "# Discharge Summary \n" at the beginning
    formatted_text = "# Discharge Summary\n\n" + formatted_text
    
    return formatted_text

In [17]:
section_names = ["Chief Complaint",
"Major Surgical or Invasive Procedure",
"History of Present Illness",
"Past Medical History",
"Social History",
"Family History",
"Physical Exam",
"Pertinent Results",
"Brief Hospital Course",
"Medications on Admission",
"Discharge Medications",
"Discharge Disposition",
"Discharge Diagnosis",
"Discharge Condition",
"Discharge Instructions",
"Followup Instructions"]

text = df.text.values[0]

In [25]:
markdown_document = format_headings_as_markdown(text, section_names)
print(markdown_document)

# Discharge Summary

 
Name:  ___                  Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: SURGERY
 
Allergies: 
Penicillins
 
Attending: ___.
 
## Chief Complaint:
Biliary pancreatitis
 
## Major Surgical or Invasive Procedure:
___: Laparoscopic cholecystectomy

 
## History of Present Illness:
Mr. ___ is a ___ man who had severe biliary 
pancreatitis resulting in pancreatic necrosis for which he was 
treated with nasojejunal feedings and pancreatic rest.  He had 
initially had multisystem organ failure, which improved. Mr. 
___ has a large postnecrotic pseudocyst, which has been 
drained through a minimally invasive approach into his GI tract. 
 He has some debris, but this is not currently infected. The 
patient was followed by Dr. ___ in his ___ 
clinic to discuss cholecystectomy. After discussion of all 
risks, benefits and possible outcomes, patient was scheduled for 
elective cholecystectomy o

In [33]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)
md_header_splits

[Document(metadata={'Header 1': 'Discharge Summary'}, page_content='Name:  ___                  Unit No:   ___  \nAdmission Date:  ___              Discharge Date:   ___  \nDate of Birth:  ___             Sex:   M  \nService: SURGERY  \nAllergies:\nPenicillins  \nAttending: ___.'),
 Document(metadata={'Header 1': 'Discharge Summary', 'Header 2': 'Chief Complaint:'}, page_content='Biliary pancreatitis'),
 Document(metadata={'Header 1': 'Discharge Summary', 'Header 2': 'Major Surgical or Invasive Procedure:'}, page_content='___: Laparoscopic cholecystectomy'),
 Document(metadata={'Header 1': 'Discharge Summary', 'Header 2': 'History of Present Illness:'}, page_content='Mr. ___ is a ___ man who had severe biliary\npancreatitis resulting in pancreatic necrosis for which he was\ntreated with nasojejunal feedings and pancreatic rest.  He had\ninitially had multisystem organ failure, which improved. Mr.\n___ has a large postnecrotic pseudocyst, which has been\ndrained through a minimally inva

In [4]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
    is_separator_regex=False,
)

In [5]:
pages = splitter.split_documents(data)
print("Number of chunks = ", len(pages))
print(pages[3].page_content)

NameError: name 'data' is not defined